# 02. 커스텀 StateGraph — 판단 루프에 UI 생성 노드 꽂기

01은 `create_agent`(프리빌트)를 썼다. 이번엔 **그래프를 직접 구성**해서
"AI 판단 → 데이터 획득 → UI 생성 → 재판단" 루프를 만든다:

```
START → agent ──(tool_calls 있음)──▶ tools ──▶ ui_builder ─┐
          ▲                                                 │
          └─────────────────────────────────────────────────┘
          └──(tool_calls 없음 = 할 일 끝)──▶ END
```

- **agent**: LLM이 대화 상태를 보고 판단 — 도구를 더 부를지, 마무리 답변을 쓸지
- **tools**: 도구 실행 (MCP)
- **ui_builder**: 도구 결과의 형태를 판별해 `{type, props}` 를 상태에 축적 ← 우리가 루프에 꽂은 커스텀 노드
- 루프는 agent 가 도구 호출을 멈출 때까지 반복 → 질문이 복합적이면 `문장-ui-문장-ui-...` 가 자연히 만들어진다

> 사전 조건: 01과 동일 — MCP 서버(`python mcp_server/server.py --http`) + Ollama `gemma4:e4b`

## 1. 준비 — MCP 도구, LLM, 파싱 함수 (01과 동일)

In [23]:
import json

from langchain_mcp_adapters.client import MultiServerMCPClient
from langchain_ollama import ChatOllama

mcp_client = MultiServerMCPClient(
    {"demo-data": {"url": "http://localhost:8000/mcp", "transport": "streamable_http"}}
)
tools = await mcp_client.get_tools()

llm = ChatOllama(model="gemma4:e4b", base_url="http://localhost:11434")


def parse_tool_output(raw) -> object:
    """MCP 결과를 벗긴다: JSON 문자열 → 객체, content block 리스트 → 내용물"""
    if isinstance(raw, str):
        try:
            return json.loads(raw)
        except json.JSONDecodeError:
            return raw
    if isinstance(raw, list):
        items = []
        for x in raw:
            if isinstance(x, dict) and x.get("type") == "text":
                items.append(parse_tool_output(x["text"]))
            else:
                items.append(parse_tool_output(x) if isinstance(x, str) else x)
        return items[0] if len(items) == 1 else items
    return raw


def build_ui_payload(data) -> dict:
    if isinstance(data, list) and data and all(isinstance(x, dict) for x in data):
        return {"type": "data-table", "props": {"columns": list(data[0].keys()), "rows": data}}
    if isinstance(data, dict) and {"city", "temp_c"} <= data.keys():
        return {"type": "weather-card", "props": data}
    return {"type": "text", "props": {"raw": str(data)[:500]}}

## 2. 상태(State) 정의

그래프의 모든 노드가 읽고 쓰는 공유 상태. **reducer** 가 핵심이다:

- `messages`: `add_messages` reducer — 노드가 반환한 메시지를 기존 목록에 병합
- `ui_events`: `operator.add` reducer — 노드는 **새 UI 만** 반환하면 그래프가 누적해준다

reducer 덕분에 노드는 "전체 상태를 다시 만드는" 게 아니라 "내가 추가할 것만" 반환한다.

In [24]:
import operator
from typing import Annotated, TypedDict

from langgraph.graph.message import add_messages


class State(TypedDict):
    messages: Annotated[list, add_messages]
    ui_events: Annotated[list, operator.add]
    comments: Annotated[list, operator.add]    # 코멘트 전용 채널 (이력과 분리)

## 3. 노드 정의

노드 = `상태를 받아 상태 변화(delta)를 반환하는 함수`. 그래프 로직이 전부 여기 드러난다.

In [36]:
from langgraph.graph import END
from langgraph.prebuilt import ToolNode

SYSTEM = (
    "너는 데이터 조회 어시스턴트다. 질문 해결에 필요한 도구를 호출하라. 도구는 한번에 하나씩만 호출하라."
    "모든 데이터가 모이면 종합 답변으로 마무리하라. 상세 수치 나열은 하지 마라."
)

llm_with_tools = llm.bind_tools(tools)


def agent_node(state: State) -> dict:
    response = llm_with_tools.invoke([("system", SYSTEM)] + state["messages"])
    if getattr(response, "tool_calls", None):
        response.tool_calls = response.tool_calls[:1]   # 턴당 도구 1개로 강제 → 루프가 잘게 돈다
    return {"messages": [response]}


# 코멘트 전용 LLM: 낮은 온도 + 출력 길이 상한 (물리적으로 나열이 불가능해짐)
comment_llm = ChatOllama(
    model="gemma4:e4b",
    base_url="http://localhost:11434",
    temperature=0.2,
    num_predict=120,
    reasoning=False,     # thinking 비활성화 → 토큰이 전부 content 로
)


def commenter_node(state: State) -> dict:
    datas = []
    for m in reversed(state["messages"]):
        if m.type != "tool":
            break
        datas.append(parse_tool_output(m.content))
    datas = list(reversed(datas))

    prompt = (
        "다음 데이터를 방금 사용자 화면에 표시했다.\n"
        "이 데이터에 대한 코멘트를 정확히 1~2문장으로 출력하라.\n"
        "금지: 선택지/옵션 제시, 번호·불릿 목록, 제목, 굵은 글씨, 인사말.\n"
        "코멘트 문장 그 자체만 출력한다.\n\n"
        + json.dumps(datas, ensure_ascii=False)
    )
    raw = comment_llm.invoke(prompt).content.strip()

    # 코드 가드: 규칙을 어겨도 첫 문단만 취하고 마크다운 장식 제거
    text = raw.split("\n\n")[0].replace("**", "").strip()
    if text.startswith(("옵션", "선택지", "#", "-", "*", "1.")):
        text = text.split(":", 1)[-1].strip()

    return {"comments": [text]} if text else {"comments": []}


def ui_builder_node(state: State) -> dict:
    new_uis = []
    for m in reversed(state["messages"]):
        if m.type != "tool":
            break
        new_uis.append(build_ui_payload(parse_tool_output(m.content)))
    return {"ui_events": list(reversed(new_uis))}


def route_after_agent(state: State) -> str:
    last = state["messages"][-1]
    return "tools" if getattr(last, "tool_calls", None) else END


tools_node = ToolNode(tools)

## 4. 그래프 조립

In [37]:
from langgraph.graph import START, StateGraph

builder = StateGraph(State)
builder.add_node("agent", agent_node)
builder.add_node("tools", tools_node)
builder.add_node("ui_builder", ui_builder_node)
builder.add_node("commenter", commenter_node)

builder.add_edge(START, "agent")
builder.add_conditional_edges("agent", route_after_agent, {"tools": "tools", END: END})
builder.add_edge("tools", "ui_builder")
builder.add_edge("ui_builder", "commenter")   # ui 직후 코멘트 강제
builder.add_edge("commenter", "agent")        # 그 뒤 재판단

graph = builder.compile()

## 5. 스트리밍 실행 — 문장과 UI 의 인터리빙

LangGraph 네이티브 스트리밍(`stream_mode` 조합)을 쓴다:

- `"messages"` — LLM 토큰 조각 실시간 (→ **token 이벤트**)
- `"updates"` — 노드가 반환한 상태 delta. `ui_builder` 의 delta 가 곧 **ui 이벤트**

agent 가 루프를 돌 때마다 `코멘트 토큰들 → ui → 코멘트 토큰들 → ui → 최종 답변` 순서가 만들어진다.

In [38]:
from pathlib import Path
from dotenv import load_dotenv

ENV_PATH = Path(r"{CHANGEME}\langfuse\.env")
assert ENV_PATH.exists(), f".env 경로 확인: {ENV_PATH}"
load_dotenv(ENV_PATH)

from langfuse import get_client
from langfuse.langchain import CallbackHandler

handler = CallbackHandler()


async def run(question: str):
    inputs = {"messages": [("user", question)], "ui_events": [], "comments": []}
    async for mode, chunk in graph.astream(
        inputs,
        stream_mode=["messages", "updates"],
        config={"callbacks": [handler]},
    ):
        if mode == "messages":
            msg_chunk, meta = chunk
            if meta.get("langgraph_node") == "agent" and msg_chunk.content:
                print(msg_chunk.content, end="", flush=True)          # 최종 답변만 토큰 스트리밍
        elif mode == "updates":
            if "ui_builder" in chunk:
                for ui in chunk["ui_builder"]["ui_events"]:
                    print(f"\n>> event: ui ({ui['type']})")
                    print(json.dumps(ui, ensure_ascii=False, indent=2))
            if "commenter" in chunk:
                for c in chunk["commenter"]["comments"]:
                    print(f"\n💬 {c}\n")                               # ui 직후 코멘트


await run("판타지 책 목록 보여주고, 서울이랑 제주 날씨도 알려줘")
get_client().flush()


>> event: ui (data-table)
{
  "type": "data-table",
  "props": {
    "columns": [
      "title",
      "author",
      "genre",
      "year",
      "rating"
    ],
    "rows": [
      {
        "title": "달빛 도서관",
        "author": "김하늘",
        "genre": "판타지",
        "year": 2021,
        "rating": 4.6
      },
      {
        "title": "시간의 정원사",
        "author": "이서준",
        "genre": "판타지",
        "year": 2019,
        "rating": 4.3
      }
    ]
  }
}

💬 두 작품 모두 판타지 장르에 속하며 비교적 높은 평점을 기록하고 있습니다. 최신작인 '달빛 도서관'이 더 높은 사용자 평가를 받고 있습니다.


>> event: ui (weather-card)
{
  "type": "weather-card",
  "props": {
    "city": "서울",
    "temp_c": 29,
    "condition": "맑음",
    "humidity": 62,
    "wind_kmh": 8
  }
}

💬 현재 서울은 기온이 29도이고 날씨가 맑으며 습도는 62%, 바람은 시속 8km로 예상됩니다. 전반적으로 화창하고 활동하기 좋은 날씨 조건입니다.


>> event: ui (weather-card)
{
  "type": "weather-card",
  "props": {
    "city": "제주",
    "temp_c": 26,
    "condition": "비",
    "humidity": 88,
    "wind_kmh": 22
  }
}

💬 현재 제주도는 비가 오고 있으며

In [35]:
from langchain.agents import create_agent

REACT_SYSTEM = (
    "너는 데이터 조회 어시스턴트다. 규칙을 반드시 지켜라:\n"
    "1. 도구는 한 번에 하나만 호출한다.\n"
    "2. 도구 결과를 받으면 그 데이터에 대해 한두 문장 코멘트를 쓴 뒤, 다음 도구를 호출한다.\n"
    "3. 모든 데이터가 모이면 종합 답변으로 마무리한다.\n"
    "4. 도구는 search_books, get_weather 만 존재한다."
)

react_agent = create_agent(llm, tools=tools, system_prompt=REACT_SYSTEM)


async def run_react(question: str):
    async for event in react_agent.astream_events({"messages": [("user", question)]}, version="v2", config={"callbacks": [handler]}):
        kind = event["event"]
        if kind == "on_chat_model_stream":
            c = event["data"]["chunk"].content
            if c:
                print(c, end="", flush=True)
        elif kind == "on_tool_end":
            data = parse_tool_output(getattr(event["data"]["output"], "content", event["data"]["output"]))
            ui = build_ui_payload(data)
            print(f"\n\n>> event: ui ({ui['type']})")
            print(json.dumps(ui, ensure_ascii=False, indent=2), "\n")


await run_react("판타지 책 목록 보여주고, 서울이랑 제주 날씨도 알려줘")



>> event: ui (data-table)
{
  "type": "data-table",
  "props": {
    "columns": [
      "title",
      "author",
      "genre",
      "year",
      "rating"
    ],
    "rows": [
      {
        "title": "달빛 도서관",
        "author": "김하늘",
        "genre": "판타지",
        "year": 2021,
        "rating": 4.6
      },
      {
        "title": "시간의 정원사",
        "author": "이서준",
        "genre": "판타지",
        "year": 2019,
        "rating": 4.3
      }
    ]
  }
} 

검색 결과, 판타지 장르의 도서로 '달빛 도서관'과 '시간의 정원사'가 있습니다. 두 책 모두 높은 평점과 흥미로운 제목을 가진 것으로 보입니다. 다음으로 서울 날씨를 확인하겠습니다.



>> event: ui (weather-card)
{
  "type": "weather-card",
  "props": {
    "city": "서울",
    "temp_c": 29,
    "condition": "맑음",
    "humidity": 62,
    "wind_kmh": 8
  }
} 



>> event: ui (weather-card)
{
  "type": "weather-card",
  "props": {
    "city": "제주",
    "temp_c": 26,
    "condition": "비",
    "humidity": 88,
    "wind_kmh": 22
  }
} 

판타지 책 목록과 서울 날씨까지 확인했습니다. 이번에는 제주도의 현재 날씨 정보입니다. 제주는 비가 오고 있으며 습도가 높고 바람이 다소

## 6. 정리 — create_agent 와의 관계

01의 `create_agent` 가 내부에서 만들어주는 것이 대략 `agent ↔ tools` 루프다.
직접 구성하면 루프 **중간에 노드를 꽂을 수 있다** — 여기서는 ui_builder 였지만 같은 자리에:

- 검증 노드 (도구 결과가 비었으면 agent 에게 재시도 지시)
- human-in-the-loop (`interrupt` — 특정 도구 호출 전 사용자 승인 대기)
- 요약 노드 (긴 대화 압축)
- checkpointer (상태 영속화 → 서버 재시작에도 대화 복원)

를 꽂는 것이 LangGraph 를 프레임워크로 쓰는 이유다.

### 관찰 과제

- [ ] 복합 질문에서 agent 가 루프를 몇 바퀴 도는가 (도구를 한 번에 다 부르는가, 나눠 부르는가)
- [ ] ui 이벤트가 토큰 스트림 사이에 끼어드는가 (인터리빙)
- [ ] SYSTEM 에서 "짧은 코멘트" 지시를 빼면 인터리빙이 어떻게 달라지는가
- [ ] route_after_agent 에 최대 루프 횟수 제한을 추가해보기 (무한 루프 방지)